In [1]:
import torch
from transformers import AutoModel, AutoTokenizer, AutoModelForSeq2SeqLM

In [ ]:
# BERT 모델 (인코더만 사용한 모델) 토크나이저, 모델 불러오기
tokenizer = AutoTokenizer.from_pretrained("klue/bert-base")
body = AutoModel.from_pretrained("klue/bert-base")
body.eval()  # 평가 모드 

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2282.77it/s]
[transformers] BertModel LOAD REPORT from: klue/bert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(32000, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
            (dropout): Dropout(p=

In [ ]:
body.config.num_hidden_layers # 설정값 확인 총 층수 12

In [6]:
# BART 모델 - 한국어 사전학습 모델을 파인튜닝(요약 전문)해서 올려둔 모델.
kobart_tokenizer = AutoTokenizer.from_pretrained("gogamza/kobart-base-v2")
kobart = AutoModelForSeq2SeqLM.from_pretrained("gogamza/kobart-summarization")
kobart.eval()  #

Loading weights: 100%|██████████| 260/260 [00:00<00:00, 1520.32it/s]


BartForConditionalGeneration(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(30000, 768, padding_idx=3)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(30000, 768, padding_idx=3)
      (embed_positions): BartLearnedPositionalEmbedding(1028, 768)
      (layers): ModuleList(
        (0-5): 6 x BartEncoderLayer(
          (self_attn): BartAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (fi

In [15]:
# 문장 토큰화
text = "나는 어제 도서관에서 책을 빌렸다."

inputs = tokenizer(text, return_tensors="pt")
tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

['[CLS]',
 '나',
 '##는',
 '어제',
 '도서관',
 '##에서',
 '책',
 '##을',
 '빌렸',
 '##다',
 '.',
 '[SEP]']

In [16]:
with torch.no_grad():
    outputs = body(**inputs)

In [ ]:
outputs.last_hidden_state.shape # (문장의 수, 토큰 수, 임베딩 차원)
# 768 개의 숫자 벡터 => representation. 모델이 문맥을 읽어 계산한 결과
# 모델은 읽지만, 사람이 해석할 수 없음.

torch.Size([1, 12, 768])

In [ ]:
# 특수 토큰
tokenizer.special_tokens_map # SEP : 구분자, CLS : 문장의 맨 앞

{'unk_token': '[UNK]',
 'sep_token': '[SEP]',
 'pad_token': '[PAD]',
 'cls_token': '[CLS]',
 'mask_token': '[MASK]'}

In [21]:
from transformers import AutoModelForMaskedLM

mlm = AutoModelForMaskedLM.from_pretrained("klue/bert-base")
mlm.eval()

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 2219.18it/s]
[transformers] BertForMaskedLM LOAD REPORT from: klue/bert-base
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(32000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, el

In [22]:
mlm.cls

BertOnlyMLMHead(
  (predictions): BertLMPredictionHead(
    (transform): BertPredictionHeadTransform(
      (dense): Linear(in_features=768, out_features=768, bias=True)
      (transform_act_fn): GELUActivation()
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    )
    (decoder): Linear(in_features=768, out_features=32000, bias=True)
  )
)

In [32]:
masked_text = f"오늘 점심으로 {tokenizer.mask_token}을 먹었다."
masked_text
inputs = tokenizer(masked_text, return_tensors="pt")
with torch.no_grad():
    outputs = mlm(**inputs)
outputs.logits.shape

torch.Size([1, 11, 32000])

In [34]:
# 32000개 중에서 점수가 1등인 위치 정보를 알려주는 것
mask_pos = (inputs["input_ids"][0] == tokenizer.mask_token_id).nonzero().item()
scores = outputs.logits[0, mask_pos] 
top_id = scores.argmax().item()
tokenizer.decode(top_id)               # argmax: 가장 높은 점수의 위치(단어 번호)

'치킨'

In [44]:
len(tokenizer.get_vocab())

32000

In [45]:
scores

tensor([-3.1608, 10.2084, -2.9104,  ..., -2.2819, -1.2834, -2.1387])

In [47]:
from transformers import AutoConfig
bert_설정 = body.config
bart_설정 = kobart.config
gpt_설정 = AutoConfig.from_pretrained("skt/kogpt2-base-v2")


In [51]:
print(f"{'':14s} {'BERT':>10s} {'GPT-2':>10s} {'BART':>12s}")
print(f"{'임베딩 차원':12s} {bert_설정.hidden_size:>10d} {gpt_설정.n_embd:>10d} {bart_설정.d_model:>12d}")
print(f"{'헤드 수':12s} {bert_설정.num_attention_heads:>10d} {gpt_설정.n_head:>10d} {bart_설정.encoder_attention_heads:>12d}")
print(f"{'층 수':12s} {bert_설정.num_hidden_layers:>10d} {gpt_설정.n_layer:>10d} {f'{bart_설정.encoder_layers}+{bart_설정.decoder_layers}':>12s}")
print(f"{'단어 수':12s} {bert_설정.vocab_size:>10d} {gpt_설정.vocab_size:>10d} {bart_설정.vocab_size:>12d}")

                     BERT      GPT-2         BART
임베딩 차원              768        768          768
헤드 수                 12         12           16
층 수                  12         12          6+6
단어 수              32000      51200        30000


In [ ]:
# 버트 모델 한 블럭의 파라미터 수 : 7087872 - 인코더
sum([ p.numel() for p in mlm.bert.encoder.layer[0].parameters() ])
# BART 모델 인코더 한 블럭의 파라미터 수 : 7087872
sum([ p.numel() for p in kobart.model.encoder.layers[0].parameters() ])
# BART 모델 디코더 한 블럭의 파라미터 수 : 9451776
sum([ p.numel() for p in kobart.model.decoder.layers[0].parameters() ])

9451776

In [ ]:
d = gpt_설정.n_embd # 768
d * d * 12 + 13 * d # GPT 모델 인코더 블록 파라미터 수도도 이론상 같다.

7087872

In [60]:
내문장들 = [                                     # ✍️ 문장을 바꿔 넣어 봐도 된다
    f"대한민국의 수도는 {tokenizer.mask_token}이다.",
    f"주말에 친구와 함께 {tokenizer.mask_token}를 봤다.",
    f"오늘 점심으로 {tokenizer.mask_token}을 먹었다.",
]

for sent in 내문장들:
    inputs = tokenizer(sent, return_tensors="pt")
    with torch.no_grad():
        outputs = mlm(**inputs)

    # [MASK] 자리 찾기 — ## 3 에서 한 것 그대로
    mask_pos = (inputs["input_ids"][0] == tokenizer.mask_token_id).nonzero().item()
    probs = outputs.logits[0, mask_pos].softmax(dim=-1)    # 32000개 점수 → 확률 (합이 1)
    top = probs.topk(5)                                    # topk(5): 큰 값 5개와 그 단어 번호

    print(sent)
    for p, idx in zip(top.values, top.indices):            # values=확률, indices=단어 번호
        print(f"   {tokenizer.decode([idx]):10s} {p.item():.3f}")
    print()

대한민국의 수도는 [MASK]이다.
   서울         0.595
   광화문        0.081
   평양         0.047
   부산         0.029
   인천         0.026

주말에 친구와 함께 [MASK]를 봤다.
   영화         0.378
   넷플릭스       0.055
   무한도전       0.055
   다큐멘터리      0.052
   드라마        0.047

오늘 점심으로 [MASK]을 먹었다.
   치킨         0.066
   [UNK]      0.057
   삼계탕        0.031
   비빔밥        0.029
   볶음밥        0.029



In [62]:
# 서브워드
tokenizer.vocab_size
for word in ["학교", "트랜스포머", "생성형AI"]:
    print(tokenizer.tokenize(word))

['학교']
['트랜스', '##포', '##머']
['생성', '##형', '##A', '##I']


In [67]:
tokenizer.tokenize("아아주맛있는삼계탕")

['아아', '##주', '##맛', '##있', '##는', '##삼', '##계', '##탕']

In [ ]:
# 시각화도구 bertviz
!uv pip install bertviz

In [70]:
from bertviz import head_view

# 어텐션 가중치를 받기 위해 설정된 모델
viz_model = AutoModel.from_pretrained("klue/bert-base",
                          output_attentions=True,
                          attn_implementation="eager"  )

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1708.89it/s]
[transformers] BertModel LOAD REPORT from: klue/bert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [71]:
from pathlib import Path
import sys

In [72]:
viz_model.eval()

배문장들 = [
    "배가 아파서 병원에 갔다.",
    "항구에서 배를 타고 섬으로 갔다.",
    "달고 시원한 배를 깎아 먹었다.",
]

for i, sent in enumerate(배문장들, start=1):
    inputs = tokenizer(sent, return_tensors="pt")
    with torch.no_grad():
        outputs = viz_model(**inputs)

    # outputs.attentions: 층마다 하나씩, 각각 (1, 헤드 12, 토큰 수, 토큰 수) — 어텐션 가중치 전부
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    # html_action="return": 화면에 바로 그리는 대신 HTML 문서로 돌려받는다 (파일로 저장하려고)
    viz_html = head_view(outputs.attentions, tokens, html_action="return")

    out_file = Path(f"bertviz_{i}.html")
    out_file.write_text(viz_html.data, encoding="utf-8")
    print(f"[{i}] {sent}  →  {out_file} 저장")

    # 노트북에서 실행 중이면 셀 아래에 바로 띄운다
    if "ipykernel" in sys.modules:
        from IPython.display import display

        display(viz_html)

[1] 배가 아파서 병원에 갔다.  →  bertviz_1.html 저장


[2] 항구에서 배를 타고 섬으로 갔다.  →  bertviz_2.html 저장


[3] 달고 시원한 배를 깎아 먹었다.  →  bertviz_3.html 저장


In [80]:
# BART 인코더-디코더 (문맥 파악 - 생성)

article = (
    "31일 외식업계에 따르면 내년도 최저임금은 1만700원으로 올해 1만320원보다 3.7% 오른다. "
    "월 209시간을 기준으로 환산한 금액은 223만6300원이다. 내수침체가 장기화해 외식경기가 "
    "어려운 가운데 최저임금까지 인상되며 외식현장에서 느끼는 인건비 압박감은 더 커질 것으로 보인다. "
    "업계에서는 인력감축, 영업시간 단축 등이 불가피할 것이란 우려가 나온다. 소상공인연합회는 "
    "\"현장을 외면한 결정\"이라며 고용노동부에 최저임금 재심의를 요청했다. 송치영 소공연 회장은 "
    "\"내년도 최저임금 인상으로 4인 고용사업장은 4대보험 부담분을 포함하면 연간 1000여만원 이상 "
    "추가부담이 발생한다\"며 \"수익감소와 경기부진이라는 이중고 속에서 소상공인발 고용축소를 "
    "앞당기는 결정\"이라고 말했다. 이에 따라 외식업계 무인·자동화현상은 더욱 가속화될 것으로 보인다. "
    "이미 주요 외식 업체를 중심으로 불고 있는 매장 무인화 바람이 더 심화하고 널리 퍼질 수 있다는 분석이다."
)

ids = kobart_tokenizer.encode(article)
input_ids = torch.tensor(
    [[kobart_tokenizer.bos_token_id] + ids + [kobart_tokenizer.eos_token_id]]
)
print(input_ids.shape)  # 입력문 토큰의 길이
summary_ids = kobart.generate(input_ids, max_length=64) # 요약문 토큰 index
print(summary_ids.shape)  # 요약문 토큰의 길이
kobart_tokenizer.decode(summary_ids[0], skip_special_tokens=True) # 요약

torch.Size([1, 198])
torch.Size([1, 32])


'31일 외식업계에 따르면 내년도 최저임금이 1만700원으로 올해 1만320원보다 3.7% 인상되면서 인건비 압박감이 커질 것으로 보인다.'